In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ================================
# 1. LOAD TABLES
# ================================
lms      = spark.table("edtech_project.edtech_bronze.lms_events_bronze")
quiz_df  = spark.table("edtech_project.edtech_bronze.quiz_attempts_bronze")
video_df = spark.table("edtech_project.edtech_bronze.video_events_bronze")
discussion = spark.table("edtech_project.edtech_silver.discussion_posts_parsed")
enrollments = spark.table("edtech_project.edtech_bronze.enrollments_bronze")

# ================================
# 2. CLEAN KEYS FUNCTION
# ================================
def clean_keys(df):
    return df.withColumn("student_id", F.trim(F.upper("student_id"))) \
             .withColumn("course_id", F.trim(F.upper("course_id")))

# ================================
# 3. VIDEO FEATURES
# ================================
video_df = video_df.withColumn(
    "video_minutes_watched",
    F.col("watched_seconds").cast("double") / 60
)

video_features = video_df.groupBy("student_id", "course_id").agg(
    F.round(F.sum("video_minutes_watched"), 2).alias("total_video_minutes_watched"),
    F.round(F.avg("completion_pct"), 2).alias("video_completion_rate")
)

video_features = clean_keys(video_features)

# ================================
# 4. QUIZ FEATURES
# ================================
quiz_df = quiz_df.withColumn("attempt_number", F.col("attempt_number").cast("int"))

quiz_level = quiz_df.groupBy("student_id", "course_id", "quiz_id").agg(
    F.max(F.when(F.col("status") == "PASSED", 1).otherwise(0)).alias("quiz_passed"),
    F.max("attempt_number").alias("attempts_per_quiz")
)

quiz_features = quiz_level.groupBy("student_id", "course_id").agg(
    F.sum("attempts_per_quiz").alias("quiz_attempts_count"),
    F.round(F.avg("quiz_passed") * 100, 2).alias("quiz_pass_rate")
)

quiz_features = clean_keys(quiz_features)

# ================================
# 5. DISCUSSION POSTS
# ================================
discussion_features = discussion.groupBy(
    "author_student_id", "course_id"
).agg(
    F.sum("student_posts").alias("discussion_posts_count")
).withColumnRenamed("author_student_id", "student_id")

discussion_features = clean_keys(discussion_features)

# ================================
# 6. ASSIGNMENT SUBMISSIONS
# ================================
lms_clean = lms.withColumn(
    "event_ts", F.col("event_ts").cast("timestamp")
)

assignment_features = lms_clean.filter(
    F.col("event_type") == "assignment_submit"
).groupBy("student_id", "course_id").agg(
    F.count("*").alias("assignment_submissions")
)

assignment_features = clean_keys(assignment_features)

# ================================
# 7. LOGIN FEATURES
# ================================
login_df = lms_clean.filter(F.col("event_type") == "login")

login_features = login_df.groupBy("student_id", "course_id").agg(
    F.sum(F.when(F.datediff(F.current_date(), F.col("event_ts")) <= 7, 1).otherwise(0)).alias("login_count_7d"),
    F.sum(F.when(F.datediff(F.current_date(), F.col("event_ts")) <= 30, 1).otherwise(0)).alias("login_count_30d")
)

login_features = clean_keys(login_features)

# ================================
# 8. LAST ACTIVE
# ================================
last_active_df = enrollments.filter(
    F.col("status").isin("ACTIVE", "PAUSED", "DROPPED")
).groupBy("student_id", "course_id").agg(
    F.max("last_accessed_date").alias("last_active_date")
)

last_active_df = last_active_df.withColumn(
    "days_since_last_active",
    F.datediff(F.current_date(), F.col("last_active_date"))
)

last_active_df = clean_keys(last_active_df)

# ================================
# 9. ENGAGEMENT TREND
# ================================
lms_weekly = lms_clean.withColumn(
    "week", F.weekofyear("event_ts")
)

weekly_counts = lms_weekly.groupBy(
    "student_id", "course_id", "week"
).count()

window_spec = Window.partitionBy("student_id", "course_id").orderBy("week")

weekly_counts = weekly_counts.withColumn(
    "prev_week_count",
    F.lag("count").over(window_spec)
).fillna({"prev_week_count": 0})

weekly_counts = weekly_counts.withColumn(
    "engagement_trend",
    F.when(F.col("count") > F.col("prev_week_count"), "IMPROVING")
     .when(F.col("count") < F.col("prev_week_count"), "DECLINING")
     .otherwise("STABLE")
)

# optional improvement
recent_weeks = weekly_counts.filter(
    F.col("week") >= F.weekofyear(F.current_date()) - 1
)
recent_weeks = clean_keys(recent_weeks)

# ================================
# 10. BASE DF (ALL UNIQUE KEYS)
# ================================
base_df = video_features.select("student_id", "course_id") \
    .union(quiz_features.select("student_id", "course_id")) \
    .union(discussion_features.select("student_id", "course_id")) \
    .union(assignment_features.select("student_id", "course_id")) \
    .union(login_features.select("student_id", "course_id")) \
    .union(last_active_df.select("student_id", "course_id")) \
    .distinct()

# ================================
# 11. FINAL JOIN
# ================================
final_df = base_df \
    .join(video_features, ["student_id", "course_id"], "left") \
    .join(quiz_features, ["student_id", "course_id"], "left") \
    .join(discussion_features, ["student_id", "course_id"], "left") \
    .join(assignment_features, ["student_id", "course_id"], "left") \
    .join(login_features, ["student_id", "course_id"], "left") \
    .join(last_active_df, ["student_id", "course_id"], "left") \
    .join(trend_df, ["student_id", "course_id"], "left")

final_df = final_df.fillna({
    "total_video_minutes_watched": 0,
    "video_completion_rate": 0,
    "quiz_attempts_count": 0,
    "quiz_pass_rate": 0,
    "discussion_posts_count": 0,
    "assignment_submissions": 0,
    "login_count_7d": 0,
    "login_count_30d": 0
})

# ================================
# 12. WRITE TO SILVER TABLE
# ================================
final_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("edtech_project.edtech_silver.shreya_checking")

In [0]:
%sql 
select * from shreya_checking limit 5;